In [ ]:
from __future__ import annotations
import os
from floquet_composite_model.src.utils.env import set_single_thread_env
from floquet_composite_model.src.utils.env import CPUPlan
set_single_thread_env(1, force=True)
import argparse
import csv
import json
import time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import h5py
import datetime as dt
today = dt.datetime.now().isoformat()
from floquet_composite_model.src.utils.partitioning import partition_slices
from floquet_composite_model.src.utils.timing import fmt_seconds
from floquet_composite_model.src.parallel.drive_sweep import worker_drive_sweep_chunk
from floquet_composite_model.src.parallel.merge import merge_axis0_chunks



In [ ]:
outdir = Path(r"C:\Users\slab\floquet\floquet\floquet_composite_model\02-16-2026_drive_sweep_v2\chunks")
files = sorted(outdir.glob("chunk_*.h5"))
def x_signature(fp):
    with h5py.File(fp, "r") as f:
        x = f["chi_ac_values"][:]
    return (len(x), float(x[0]), float(x[-1]))
sigs = {}
for fp in files:
    sigs.setdefault(x_signature(fp), []).append(fp.name)
for sig, names in sorted(sigs.items()):
    print(sig, len(names), names)

(300, 0.0, 1.2566370614359172) 43 ['chunk_0000.h5', 'chunk_0001.h5', 'chunk_0002.h5', 'chunk_0003.h5', 'chunk_0004.h5', 'chunk_0005.h5', 'chunk_0006.h5', 'chunk_0007.h5', 'chunk_0008.h5', 'chunk_0009.h5', 'chunk_0010.h5', 'chunk_0011.h5', 'chunk_0012.h5', 'chunk_0013.h5', 'chunk_0014.h5', 'chunk_0015.h5', 'chunk_0016.h5', 'chunk_0017.h5', 'chunk_0018.h5', 'chunk_0019.h5', 'chunk_0020.h5', 'chunk_0021.h5', 'chunk_0022.h5', 'chunk_0023.h5', 'chunk_0024.h5', 'chunk_0025.h5', 'chunk_0026.h5', 'chunk_0027.h5', 'chunk_0028.h5', 'chunk_0029.h5', 'chunk_0030.h5', 'chunk_0031.h5', 'chunk_0032.h5', 'chunk_0033.h5', 'chunk_0034.h5', 'chunk_0035.h5', 'chunk_0036.h5', 'chunk_0037.h5', 'chunk_0038.h5', 'chunk_0039.h5', 'chunk_0040.h5', 'chunk_0041.h5', 'chunk_0042.h5']


In [ ]:
x_key = "chi_ac_values"
y_key = "omega_d_values"
data_keys = ["scar_state0", "scar_state1"]
extra_keys = [] #like["_timing"]

def y0(fp):
    with h5py.File(fp, "r") as f:
        return float(f[y_key][0])

files = sorted(files, key=y0)

with h5py.File(files[0], "r") as f0:
    x0 = f0[x_key][:]
    meta_attrs = {}
    meta_datasets = {}
    if "_metadata" in f0:
        meta_group = f0["_metadata"]
        meta_attrs = dict(meta_group.attrs.items())
        meta_datasets = {k: meta_group[k][:] for k in meta_group.keys()}

ys = []
data = {k: [] for k in data_keys + extra_keys}

for fp in files:
    with h5py.File(fp, "r") as f:
        x = f[x_key][:]
        if len(x) != len(x0) or not np.allclose(x, x0):
            raise ValueError(f"x grid mismatch in {fp}")
        ys.append(f[y_key][:])
        for k in data_keys:
            data[k].append(f[k][:])
        for k in extra_keys:
            if k in f:
                data[k].append(f[k][:])

y_all = np.concatenate(ys, axis=0)
out_path = outdir.parent / "merged.h5"

with h5py.File(out_path, "w") as out:
    out.create_dataset(x_key, data=x0)
    out.create_dataset(y_key, data=y_all)
    for k in data_keys:
        out.create_dataset(
            k,
            data=np.concatenate(data[k], axis=0),
            compression="gzip",
            compression_opts=4,
            shuffle=True,
        )
    for k in extra_keys:
        if data[k]:
            out.create_dataset(k, data=np.concatenate(data[k], axis=0))
    if meta_attrs or meta_datasets:
        g = out.create_group("_metadata")
        for ak, av in meta_attrs.items():
            g.attrs[ak] = av
        for dk, dv in meta_datasets.items():
            g.create_dataset(dk, data=dv)

print("Wrote:", out_path)

Wrote: C:\Users\slab\floquet\floquet\floquet_composite_model\02-16-2026_drive_sweep_v2\merged.h5
